In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)
warnings.filterwarnings("ignore", message="The parameter 'pretrained' is deprecated since 0.13.*", category=UserWarning)
warnings.filterwarnings("ignore", message="Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13.*", category=UserWarning)
from panel_segmentation import panel_detection as pseg
from panel_segmentation.env import load_env_file, require_env
import numpy as np
from tensorflow.keras.preprocessing import image as imagex
import matplotlib.pyplot as plt
import os
import pandas as pd

_ = load_env_file()

# Panel Detection

The following example uses the panel detection pipeline to detect solar panels in satellite imagery and get their mounting configuration and azimuth. Let's use an NREL site as an example and get its latitude-longitude coordinates.

In [ ]:
#Example latitude-longitude coordinates to run the analysis on.
#AVSR1: 34.7803, -118.4244
#latitude = 39.7407
#longitude = -105.1694
latitude = 34.7803
longitude = -118.4244
google_maps_api_key = require_env("GOOGLE_MAPS_API_KEY")
output_dir = "../../../outputs/panel_detection_examples"
os.makedirs(output_dir, exist_ok=True)
file_name_save = os.path.join(output_dir, "sat_img_ex.png")

Create an instance of the PanelDetection class and generate a satellite image of the site at the lat-long coordinates. 

In [ ]:
#CREATE AN INSTANCE OF THE PANELDETECTION CLASS TO RUN THE ANALYSIS
panelseg = pseg.PanelDetection()

#GENERATE A SATELLITE IMAGE USING THE ASSOCIATED LAT-LONG COORDS AND THE GOOGLE
#MAPS API KEY
img = panelseg.generateSatelliteImage(latitude, longitude,
                                      file_name_save,
                                      google_maps_api_key)
#Show the generated satellite image
plt.imshow(img)

Load in the image and declare it as a numpy array.

In [ ]:
x = imagex.load_img(
    file_name_save,
    color_mode='rgb', target_size=(640,640))
plt.imshow(x)
x = np.array(x)

Use the classifier model to confirm if there are solar arrays detected in the satellite image.

In [ ]:
panel_loc = panelseg.hasPanels(x)
print(panel_loc)

Classify the image by mounting configuration, using the mounting configuration object detection algorithm.

In [ ]:
(scores, labels, boxes) = panelseg.classifyMountingConfiguration(
    image_file_path = file_name_save,
    acc_cutoff = .65)

First, mask the satellite image, and then crop out the panels.

In [ ]:
#Mask the satellite image
res = panelseg.testSingle(x.astype(float), test_mask=None,  model =None)    
#Use the mask to isolate the panels
new_res = panelseg.cropPanels(x, res)
plt.imshow(new_res.reshape(640,640,3))

In [ ]:
import cv2
rest = cv2.cvtColor(res,cv2.COLOR_GRAY2RGB)
#Cluster the solar arrays in the image using connected components clustering.
n,clusters = panelseg.clusterPanels(new_res,
                                    boxes,
                                    fig=True)


In [ ]:
#Calculate the azimuth for each cluster
azimuth_values = []
for ii in np.arange(clusters.shape[0]):
    az = panelseg.detectAzimuth(clusters[ii][np.newaxis,:])
    azimuth_values.append(float(az))
    print(az)

#Plot the azimuth estimate for each cluster
panelseg.plotEdgeAz(clusters, 3, 1,
                    save_img_file_path = output_dir)

In [ ]:
cluster_summary = pd.DataFrame({
    'cluster': np.arange(1, len(azimuth_values) + 1),
    'mounting_type_label': labels,
    'azimuth_value': azimuth_values,
})
cluster_summary

Now let's run the complete pipeline function on an example site:

In [ ]:
panelseg.runSiteAnalysisPipeline(
    file_name_save_img=file_name_save,
    file_name_save_mount=os.path.join(output_dir,
                                     "mounting_predictions.png"),
    file_path_save_azimuth=output_dir)